# Asking the graph to do arithmetic

The graph is built by an LLM reading the SysML, plus a lexer reading the same files
for the two things the syntax states outright: attribute values, and the
containment and typing that a rollup has to walk. This notebook is only the
quantitative half -- the questions that need exact numbers and an exact tree.

Every answer below is AQLizer writing its own AQL. Nothing is hand-queried.

In [1]:
import logging
from sysml import nl
logging.disable(logging.INFO)

## The benchmark

Sum the dry mass of the Saturn V from its stages. Nobody wrote the total down; the
stages are three levels below the vehicle, and the masses sit on the part
*definitions* rather than on the usages the vehicle contains.

Before the lexer step this returned nothing at all -- the LLM had extracted no
containment edge from `SaturnV` to anything, and `S-IC` had no relations of any
kind.

In [2]:
nl.instance().ask(
    "What is the total dry mass of the Saturn V, summed from its stages?").show()

Q  What is the total dry mass of the Saturn V, summed from its stages?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.entity_name == "SATURNV"
     LET parts = (
       FOR child, edge IN 1..6 OUTBOUND e sysml_Relations
         FILTER edge.relationship_type IN ["owns", "typedby"]
         FILTER child.attributes.dryMass.value != null
         RETURN DISTINCT {name: child.entity_name, mass: child.attributes.dryMass.value,
                          unit: child.attributes.dryMass.unit,
                          at: CONCAT(child.source_file, ":", child.source_line)})
     RETURN {total: SUM(parts[*].mass), unit: "kg", contributors: parts}

rows (1, first 1)
   {"total": 188650, "unit": "kg", "contributors": [{"name": "SATURNVINSTRUMENTUNIT", "mass": 1950, "unit": "kg", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:44"}, {"name": "S-IVB", "mass": 13500, "unit": "kg", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage

## The same shape, over money

Four cost attributes on the mission, in dollars.

In [3]:
nl.instance().ask(
    "What did the Apollo 11 mission cost in total, and what are the parts of that "
    "figure?").show()

Q  What did the Apollo 11 mission cost in total, and what are the parts of that figure?

AQL
   FOR e IN sysml_Entities
     FILTER e.entity_name == "APOLLO11MISSION"
     LET costs = (
       FOR name IN ATTRIBUTES(e.attributes)
         FILTER LIKE(LOWER(name), "%cost%") AND e.attributes[name].value != null
         RETURN {name, value: e.attributes[name].value, unit: e.attributes[name].unit}
     )
     RETURN {total: SUM(costs[*].value), currency: "$", parts: costs}

rows (1, first 1)
   {"total": 11000000000.0, "currency": "$", "parts": [{"name": "manufacturingCost", "value": 3000000000, "unit": "$"}, {"name": "operationsCost", "value": 1000000000, "unit": "$"}, {"name": "researchAndDevelopmentCost", "value": 5000000000, "unit": "$"}, {"name": "personnelCost", "value": 2000000000, 

A  The total cost of the Apollo 11 mission was $11 billion. This total comprises the following parts: $3 billion for manufacturing costs, $1 billion for operations costs, $5 billion for research and de

## A few more

Heaviest things in the model, wherever the mass is written down.

In [4]:
nl.instance().ask(
    "Which ten elements have the largest mass or dry mass, with units and the "
    "file and line each is declared on?").show(row_limit=10)

Q  Which ten elements have the largest mass or dry mass, with units and the file and line each is declared on?

AQL
   WITH sysml_Entities
   FOR e IN sysml_Entities
     FILTER e.attributes != null
     FOR name IN ATTRIBUTES(e.attributes)
       LET a = e.attributes[name]
       FILTER (name == "mass" OR name == "dryMass") AND a.value != null
       SORT a.value DESC
       LIMIT 10
       RETURN {
         element: e.entity_name, 
         attribute: name, 
         value: a.value, 
         unit: a.unit,
         at: CONCAT(e.source_file, ":", e.source_line)
       }

rows (10, first 10)
   {"element": "S-IC", "attribute": "dryMass", "value": 137000, "unit": "kg", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:217"}
   {"element": "S-II", "attribute": "dryMass", "value": 36200, "unit": "kg", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:201"}
   {"element": "S-IVB", "attribute": "dryMass", "value": 13500, "unit": "kg", "at": "apollo-11-s

Reliability figures, which are on a different attribute and in a different unit.

In [5]:
nl.instance().ask(
    "Which components have a failure rate, and what is it?").show(row_limit=8)

Q  Which components have a failure rate, and what is it?

AQL
   WITH sysml_Entities
   FOR e IN sysml_Entities
     FILTER e.attributes.failureRate.value != null
     RETURN {element: e.entity_name, value: e.attributes.failureRate.value, unit: e.attributes.failureRate.unit, at: CONCAT(e.source_file, ":", e.source_line)}

rows (6, first 6)
   {"element": "TECHNICALCOMPONENTSPACKAGE_LUNARMODULEDESCENTSTAGE", "value": 3e-06, "unit": "1/h", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:111"}
   {"element": "TECHNICALCOMPONENTSPACKAGE_LUNARMODULEASCENTSTAGE", "value": 2.5e-06, "unit": "1/h", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:124"}
   {"element": "SATURNVINSTRUMENTUNIT", "value": 5e-06, "unit": "1/h", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:44"}
   {"element": "APOLLOLAUNCHESCAPESYSTEM", "value": 1e-07, "unit": "1/h", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:54"}
   {"element":

A gap question -- no arithmetic, but it needs the same exact edges.

In [6]:
nl.instance().ask(
    "Which requirements in the DroneModelLogical model does nothing satisfy?").show(row_limit=8)

Q  Which requirements in the DroneModelLogical model does nothing satisfy?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.entity_type == 'requirement' AND 'DroneModelLogical' IN e.models
     LET satisfied = LENGTH(
       FOR r IN sysml_Relations
         FILTER r._to == e._id AND r.relationship_type == 'satisfies'
         RETURN 1)
     FILTER satisfied == 0
     RETURN {requirement: e.entity_name, files: e.files}

rows (41, first 8)
   {"requirement": "DE-REQ-3 DURABILITY", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "ENGINEEFFICIENCY", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "REQUIREMENTPOWER", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "DE-REQ-10 COSTEFFECTIVE", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "DURABILITY", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "DE-REQ-9 SAFETY", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "DE-REQ-8 COMPATIBILITY", "files": [

And one that crosses the two vehicles, over the computed similarity layer rather
than anything either file states.

In [7]:
nl.instance().ask(
    "What in the Apollo model plays a role like the drone's engines?").show(row_limit=5)

Q  What in the Apollo model plays a role like the drone's engines?

AQL
   WITH sysml_Chunks, sysml_Communities, sysml_Documents, sysml_Entities
   FOR e IN sysml_Entities
     FILTER CONTAINS(e.entity_name, "ENGINES")
     FILTER 'DroneModelLogical' IN e.models
     FOR v, r IN 1..1 ANY e sysml_Relations
       FILTER r.type == 'SIMILAR_TO'
       FILTER 'apollo-11-sysml-v2' IN v.models
       SORT r.cosine DESC
       RETURN {element: e.entity_name, counterpart: v.entity_name, models: v.models, role: r.analogy_role, cosine: r.cosine}

rows (10, first 5)
   {"element": "ENGINES", "counterpart": "S-IC_ENGINES", "models": ["apollo-11-sysml-v2"], "role": "part", "cosine": 0.6872469562485976}
   {"element": "ENGINES", "counterpart": "S-II_ENGINES", "models": ["apollo-11-sysml-v2"], "role": "part", "cosine": 0.6857752088769634}
   {"element": "ENGINES", "counterpart": "S-IVB_ENGINES", "models": ["apollo-11-sysml-v2"], "role": "part", "cosine": 0.6754204066913535}
   {"element": "DRONEENGIN

## It is the grammar, not these files

The lexer knows SysML v2's declaration syntax, not this corpus. Here it is on a
file it has never seen, using forms the three models do not contain: a typed
attribute, scientific notation, an expression instead of a value, a quoted name
with spaces, a short name, two supertypes at once.

In [8]:
from sysml.pipeline import structure

SRC = """
package NewVehicle {
    doc /* Documentation, which must not be read as a declaration. */
    part def <'PV-1'> PoweredVehicle :> Vehicle, Machine {
        attribute mass : ISQ::MassValue = 42.5 [kg];
        attribute :>> topSpeed = 3.6E2 [km/h];
        attribute derivedMass = mass * 2;
        part engine : CombustionEngine;
    }
    part def 'Odd Name With Spaces' { attribute cost = 1200 ['$']; }
    enum def Colour { red; green; }
    part myCar : PoweredVehicle;
}
"""
elements, relations = {}, []
structure.walk("unseen.sysml", SRC, elements, relations)
for e in elements.values():
    print(f"{e['kind']:<12} {e['qualified']:<34} {e.get('attributes') or ''}")
print()
for r in relations:
    print(f"{r['type']:<12} {r['from']}  ->  {r['to']}")

package      NewVehicle                         
part         NewVehicle::PoweredVehicle         {'mass': {'value': 42.5, 'unit': 'kg'}, 'topSpeed': {'value': 360, 'unit': 'km/h'}, 'derivedMass': {'expression': 'mass * 2'}}
part         NewVehicle::PoweredVehicle::engine 
part         NewVehicle::Odd Name With Spaces   {'cost': {'value': 1200, 'unit': '$'}}
enumeration  NewVehicle::Colour                 
part         NewVehicle::myCar                  

owns         NewVehicle  ->  NewVehicle::PoweredVehicle
specializes  NewVehicle::PoweredVehicle  ->  Vehicle
specializes  NewVehicle::PoweredVehicle  ->  Machine
owns         NewVehicle::PoweredVehicle  ->  NewVehicle::PoweredVehicle::engine
typedby      NewVehicle::PoweredVehicle::engine  ->  CombustionEngine
owns         NewVehicle  ->  NewVehicle::Odd Name With Spaces
owns         NewVehicle  ->  NewVehicle::Colour
owns         NewVehicle  ->  NewVehicle::myCar
typedby      NewVehicle::myCar  ->  PoweredVehicle


## What is exact and what is not

`attributes`, `source_file`, `source_line`, and the `owns` / `typedby` /
`specializes` edges are read straight out of the syntax -- they carry `stated:
true` and are as reliable as the file. Everything else on the graph is the LLM's
reading of the prose: the descriptions, and the `satisfies` / `refines` /
`dependson` relations that no keyword expresses.

The split is deliberate. The numbers and the tree have to be right, and an LLM is
not reliable about them -- on this corpus it produced 68 containment edges where
the files state about seventeen hundred. What it is good at is the half a lexer
cannot reach, which is why both passes are here.

In [9]:
from sysml import config
db = config.db()
Q = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "RELATED_TO"
  COLLECT stated = r.stated == true WITH COUNT INTO n
  RETURN {{source: stated ? "read from the syntax" : "inferred by the LLM", n}}'''
for row in db.aql.execute(Q):
    print(f"{row['n']:>6}  {row['source']}")

   934  inferred by the LLM
  3123  read from the syntax
